In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\luis\Documents\TFG - Data-Centric AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
from IPython.display import display

from utils.thresholds.quality import (
    ensure_quality_metric,
    evaluate_quality_thresholds_against_manual_labels,
    export_quality_review_images,
    get_quality_filter_spec,
    recommend_quality_threshold,
    sample_images_for_quality_threshold_review,
)

FILTER_NAME = "uniformity"
METADATA_PATH = PROJECT_ROOT / "data" / "phase2" / "phase2_train.csv"
IMAGES_DIR = PROJECT_ROOT / "data" / "phase2" / "frames"

REPORTS_DIR = PROJECT_ROOT / "reports" / f"{FILTER_NAME}_threshold_selection"
REVIEW_IMAGES_DIR = REPORTS_DIR / "review_images"
REVIEW_CSV_PATH = REPORTS_DIR / "manual_quality_review.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

spec = get_quality_filter_spec(FILTER_NAME)
METRIC_COLUMN = spec["metric_column"]
THRESHOLD_CANDIDATES = spec["thresholds"]
N_REVIEW_IMAGES = 100
N_REVIEW_IMAGES_PER_THRESHOLD = N_REVIEW_IMAGES // len(THRESHOLD_CANDIDATES)

print("FILTER_NAME:", FILTER_NAME)
print("METADATA_PATH:", METADATA_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("THRESHOLD_CANDIDATES:", THRESHOLD_CANDIDATES)

In [ ]:
df = pd.read_csv(METADATA_PATH)
df = ensure_quality_metric(
    dataframe=df,
    filter_name=FILTER_NAME,
    images_dir=IMAGES_DIR,
)

print("Images:", len(df))
display(df[["filename", METRIC_COLUMN]].head())
display(df[METRIC_COLUMN].describe())

In [ ]:
review_df = sample_images_for_quality_threshold_review(
    dataframe=df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    n_per_threshold=N_REVIEW_IMAGES_PER_THRESHOLD,
)

review_df = export_quality_review_images(
    review_df=review_df,
    images_dir=IMAGES_DIR,
    output_dir=REVIEW_IMAGES_DIR,
    image_size=(320, 180),
)

review_df.to_csv(REVIEW_CSV_PATH, index=False)

print("CSV to label:", REVIEW_CSV_PATH)
print("Images exported to:", REVIEW_IMAGES_DIR)
display(review_df.head())

In [ ]:
labeled_df = pd.read_csv(REVIEW_CSV_PATH, sep=None, engine="python")
labeled_df["manual_label"] = (
    labeled_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {"discard", "valuable", "uncertain"}
invalid_labels = set(labeled_df["manual_label"].unique()) - valid_labels

if invalid_labels:
    raise ValueError(f"Invalid labels: {invalid_labels}")

label_counts = (
    labeled_df["manual_label"]
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="count")
)

display(label_counts)

In [ ]:
manual_evaluation_df = evaluate_quality_thresholds_against_manual_labels(
    labeled_df=labeled_df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    positive_labels=("discard",),
    negative_labels=("valuable",),
)

manual_evaluation_df = manual_evaluation_df.sort_values(
    ["precision", "recall", "false_positive"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(manual_evaluation_df)

In [ ]:
recommended_df = recommend_quality_threshold(
    evaluation_df=manual_evaluation_df,
    min_precision=0.90,
)

display(recommended_df)

selected = recommended_df.iloc[0]
FINAL_THRESHOLD = float(selected["threshold"])

print(f"Selected {FILTER_NAME} threshold: {FINAL_THRESHOLD:g}")